# DeepAgent Quickstart — Azure AI Foundry

Uses `deepagents` with an **Azure OpenAI** model deployed via [Azure AI Foundry](https://ai.azure.com/).

### Prerequisites
Set the following environment variables (or put them in a `.env` file):

```bash
AZURE_OPENAI_API_KEY=<your-key>
AZURE_OPENAI_ENDPOINT=https://<your-resource>.openai.azure.com/
AZURE_OPENAI_DEPLOYMENT_NAME=<your-deployment-name>   # e.g. gpt-4o
AZURE_OPENAI_API_VERSION=2025-04-01-preview
```

In [13]:
# Install dependencies (skip if already installed)
%pip install -q deepagents langchain-openai python-dotenv --prerelease=allow

/Users/bhaiya.singh/Documents/My Repos/LangChain-DeepAgent/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [29]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)  # loads from .env if present

# Verify required env vars are set
required = [
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_DEPLOYMENT_NAME",
    "AZURE_OPENAI_API_VERSION",
]
missing = [k for k in required if not os.getenv(k)]
if missing:
    raise EnvironmentError(f"Missing env vars: {missing}")

print("✅ Environment configured")

✅ Environment configured


In [37]:
from langchain_openai import AzureChatOpenAI

# Initialise the Azure OpenAI model
model = AzureChatOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    azure_deployment=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
    openai_api_version=os.environ["AZURE_OPENAI_API_VERSION"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    temperature=0,
)

print(f"✅ Model ready: {model}")

✅ Model ready: metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.10', 'langchain-openai': '1.3.2'}} output_version=None profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True} client=<openai.resources.chat.completions.completions.Completions object at 0x117264050> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x1172647c0> root_client=<openai.lib.azure.AzureOpenAI object 

In [38]:
model.invoke("hi")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 8, 'total_tokens': 18, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 9, 'engine_ttft_ms': 32, 'engine_ttlt_ms': 118, 'pre_inference_ms': 91, 'service_tbt_ms': 9, 'service_ttft_ms': 342, 'service_ttlt_ms': 431, 'total_duration_ms': 343, 'user_visible_ttft_ms': 251}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_965c8b9ecf', 'id': 'chatcmpl-DtRgC3smjDqfT4TpxEwmPtzI0Temp', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_h

In [39]:
# Define custom tools — plain Python functions with docstrings

def get_weather(city: str) -> str:
    """Return the current weather for a given city."""
    # Replace with a real weather API call in production
    return f"The weather in {city} is 22°C and sunny."


def calculate(expression: str) -> str:
    """Safely evaluate a simple mathematical expression and return the result."""
    allowed = set("0123456789+-*/.() ")
    if not all(c in allowed for c in expression):
        return "Error: only basic arithmetic is supported."
    try:
        return str(eval(expression))  # noqa: S307 — guarded by allowlist above
    except Exception as e:
        return f"Error: {e}"


print("✅ Tools defined")

✅ Tools defined


In [40]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model=model,
    tools=[get_weather, calculate],
    system_prompt=(
        "You are a helpful assistant powered by Azure AI Foundry. "
        "Use your tools whenever they help answer the user's question."
    ),
)

print("✅ DeepAgent created")

✅ DeepAgent created


In [41]:
# --- Single invocation ---
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the weather in Dubai, and what is 123 * 456?"}]}
)

print(result["messages"][-1].content)

The weather in Dubai is 22°C and sunny. The result of 123 * 456 is 56,088.
